# AI Logistics Command Center — Real Olist Data

This notebook downloads the official Brazilian E-Commerce Public Dataset by Olist from Kaggle and runs the repository AI for freight prediction, late-delivery risk, nearest-seller routing, and carrier selection.

The official dataset contains about 100k orders and also includes a geolocation table linking Brazilian ZIP-code prefixes to latitude/longitude observations. citeturn819213search0turn819213search1

Important: Olist does not contain historical shipping-company/carrier records. Carrier selection therefore uses the configurable carrier table in the repository. Replace its placeholder commercial parameters with real carrier contracts and measured performance before operational use.\n

In [ ]:
!pip -q install -U kagglehub pandas numpy scikit-learn joblib\n

## 1. Kaggle access

KaggleHub can download the public dataset directly. When authentication is required, store the Kaggle API token as a Colab Secret named KAGGLE_API_TOKEN.\n

In [ ]:
import os
from pathlib import Path

try:
    from google.colab import userdata
    token = userdata.get('KAGGLE_API_TOKEN')
    if token:
        os.environ['KAGGLE_API_TOKEN'] = token
        print('Kaggle token loaded from Colab Secret.')
    else:
        print('No Kaggle token found; trying public dataset access.')
except Exception:
    print('Colab Secrets unavailable; trying public dataset access.')\n

In [ ]:
import kagglehub

download_root = Path('/content/olist_data')
download_root.mkdir(parents=True, exist_ok=True)

dataset_path = kagglehub.dataset_download(
    'olistbr/brazilian-ecommerce',
    output_dir=str(download_root),
)
print('Downloaded to:', dataset_path)\n

In [ ]:
required_files = [
    'olist_orders_dataset.csv',
    'olist_order_items_dataset.csv',
    'olist_customers_dataset.csv',
    'olist_products_dataset.csv',
    'olist_sellers_dataset.csv',
    'olist_geolocation_dataset.csv',
]

matches = {}
for name in required_files:
    found = list(download_root.rglob(name))
    if not found:
        raise FileNotFoundError(f'Missing {name}')
    matches[name] = found[0]

data_dir = matches['olist_orders_dataset.csv'].parent
print('Real Olist data directory:', data_dir)
for p in sorted(data_dir.glob('*.csv')):
    print(f'{p.name:45s} {p.stat().st_size / 1024 / 1024:8.2f} MB')\n

## 2. Load the real datasets\n

In [ ]:
import pandas as pd

orders = pd.read_csv(data_dir / 'olist_orders_dataset.csv')
items = pd.read_csv(data_dir / 'olist_order_items_dataset.csv')
customers = pd.read_csv(data_dir / 'olist_customers_dataset.csv')
products = pd.read_csv(data_dir / 'olist_products_dataset.csv')
sellers = pd.read_csv(data_dir / 'olist_sellers_dataset.csv')
geo = pd.read_csv(data_dir / 'olist_geolocation_dataset.csv')

print('orders:', orders.shape)
print('items:', items.shape)
print('customers:', customers.shape)
print('products:', products.shape)
print('sellers:', sellers.shape)
print('geolocation:', geo.shape)\n

In [ ]:
display(orders.head(3))
display(sellers.head(3))
display(geo.head(3))\n

## 3. Business insights from the real data

The KPI calculation below uses a distinct order-level aggregation so multi-item orders are not accidentally weighted multiple times.\n

In [ ]:
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date',
]
for col in date_cols:
    orders[col] = pd.to_datetime(orders[col], errors='coerce')

items['price'] = pd.to_numeric(items['price'], errors='coerce')
items['freight_value'] = pd.to_numeric(items['freight_value'], errors='coerce')

order_money = (
    items.groupby('order_id', as_index=False)
    .agg(
        product_revenue=('price', 'sum'),
        freight_cost=('freight_value', 'sum'),
        item_count=('order_item_id', 'count'),
    )
)

business = orders.merge(order_money, on='order_id', how='left')
business['total_sales'] = business['product_revenue'] + business['freight_cost']
delivered = business[
    business['order_delivered_customer_date'].notna()
    & business['order_estimated_delivery_date'].notna()
].copy()
delivered['delivery_days'] = (
    delivered['order_delivered_customer_date']
    - delivered['order_purchase_timestamp']
).dt.total_seconds() / 86400
delivered['days_late'] = (
    delivered['order_delivered_customer_date']
    - delivered['order_estimated_delivery_date']
).dt.total_seconds() / 86400
delivered['is_late'] = (delivered['days_late'] > 0).astype(int)

revenue = business['product_revenue'].sum()
freight = business['freight_cost'].sum()
late_rate = delivered['is_late'].mean() * 100
avg_delivery = delivered['delivery_days'].mean()

kpis = pd.DataFrame({
    'KPI': [
        'Product revenue', 'Freight cost', 'Freight / product revenue',
        'Orders', 'Unique customers', 'Avg delivery days', 'Late delivery %',
    ],
    'Value': [
        revenue, freight, freight / revenue * 100,
        business['order_id'].nunique(), customers['customer_unique_id'].nunique(),
        avg_delivery, late_rate,
    ],
})
display(kpis)\n

## 4. Load your GitHub branch

The notebook pulls the current AI implementation from the ai-freight-delivery branch so the Colab run matches the project code.\n

In [ ]:
import subprocess, shutil

repo_dir = Path('/content/Executive-Sales-Analytics-Dashboard')
if repo_dir.exists():
    shutil.rmtree(repo_dir)

subprocess.run([
    'git', 'clone', '-b', 'ai-freight-delivery', '--depth', '1',
    'https://github.com/maroamratef/Executive-Sales-Analytics-Dashboard.git',
    str(repo_dir),
], check=True)
print(repo_dir)\n

## 5. Train the AI on the real Olist data\n

In [ ]:
ai_output = repo_dir / 'ai_output'
training = subprocess.run([
    'python', str(repo_dir / 'python' / 'ai_freight_delivery.py'),
    '--data-dir', str(data_dir),
    '--output-dir', str(ai_output),
], text=True, capture_output=True)
print(training.stdout)
if training.returncode != 0:
    print(training.stderr)
    raise RuntimeError('AI training failed.')\n

In [ ]:
import json
summary = json.loads((ai_output / 'model_summary.json').read_text())
print(json.dumps(summary, indent=2))\n

In [ ]:
print('Freight feature importance')
display(pd.read_csv(ai_output / 'freight_feature_importance.csv').head(15))
print('Late-delivery feature importance')
display(pd.read_csv(ai_output / 'late_delivery_feature_importance.csv').head(15))\n

## 6. Automatic nearest-seller routing

The optimizer now automatically uses the official Olist geolocation dataset. It averages repeated ZIP-prefix observations into centroids and calculates Haversine distance between customer and seller locations. This is a ZIP-centroid distance, not doorstep GPS distance. citeturn819213search0turn819213search2\n

## 7. Shipping-company configuration

Olist does not provide carrier history. The repository carrier table is therefore a configurable interface for your real shipping companies, prices, capacities, and service-level data. The routing logic itself is connected to the real Olist orders/products/customers/sellers/geolocation data.\n

In [ ]:
carrier_file = repo_dir / 'config' / 'carriers.csv'
display(pd.read_csv(carrier_file))\n

## 8. Run automated seller + carrier assignment\n

In [ ]:
routing = subprocess.run([
    'python', str(repo_dir / 'python' / 'logistics_optimizer.py'),
    '--data-dir', str(data_dir),
    '--models-dir', str(ai_output),
    '--carriers', str(carrier_file),
    '--limit', '20',
], text=True, capture_output=True)
print(routing.stdout)
if routing.returncode != 0:
    print(routing.stderr)
    raise RuntimeError('Routing failed.')\n

In [ ]:
assignments = pd.read_csv(ai_output / 'automated_logistics_assignments.csv')
display(assignments.head(20))

if 'selected_seller_id' in assignments.columns:
    print('Successful assignments:', assignments['selected_seller_id'].notna().sum())
print('Rows processed:', len(assignments))\n

## 9. Inspect one real order\n

In [ ]:
single_product_orders = (
    items.groupby('order_id')['product_id']
    .nunique()
    .sort_values()
)
demo_order_id = single_product_orders.index[single_product_orders.eq(1)][0]
print('Demo order:', demo_order_id)

one = subprocess.run([
    'python', str(repo_dir / 'python' / 'logistics_optimizer.py'),
    '--data-dir', str(data_dir),
    '--models-dir', str(ai_output),
    '--carriers', str(carrier_file),
    '--order-id', str(demo_order_id),
], text=True, capture_output=True)
print(one.stdout)
if one.returncode != 0:
    print(one.stderr)
    raise RuntimeError('Single-order routing failed.')

assignments = pd.read_csv(ai_output / 'automated_logistics_assignments.csv')
display(assignments[assignments['order_id'].astype(str).eq(str(demo_order_id))])\n

In [ ]:
carrier_options = pd.read_csv(ai_output / 'carrier_options_by_order.csv')
display(carrier_options[carrier_options['order_id'].astype(str).eq(str(demo_order_id))])\n

## 10. Result

For each real Olist order, the pipeline produces a prospective routing decision containing the selected seller, geographic distance, predicted freight, late-delivery risk, selected carrier, estimated carrier cost, estimated ETA, and the decision explanation.

Actual shipment booking or rerouting requires API connections to real sellers, order-management software, and shipping companies. The notebook demonstrates the complete analytical assignment layer using the real Olist dataset.\n

In [ ]:
print('Generated AI outputs:')
for p in sorted(ai_output.iterdir()):
    print(p.name)\n